In [15]:
import os
import io
import glob
import pandas as pd
import numpy as np
from scipy.spatial import KDTree

def read_well_data(well_file_path):
    """
    Безопасное считывание файла скважин. Удаляет текстовые заголовки, если они есть.
    """
    # Читаем все колонки как текст, чтобы не упасть на заголовках
    df_wells = pd.read_csv(well_file_path, sep=r'\s+', names=['X', 'Y', 'WellName', 'Heff'], dtype=str)
    
    # Принудительно переводим координаты в числа. Заголовки (типа 'X', 'Y') превратятся в NaN
    df_wells['X'] = pd.to_numeric(df_wells['X'], errors='coerce')
    df_wells['Y'] = pd.to_numeric(df_wells['Y'], errors='coerce')
    df_wells['Heff'] = pd.to_numeric(df_wells['Heff'], errors='coerce')
    
    # Удаляем строки, где X или Y не стали числами (т.е. бывшие строки заголовков)
    df_wells = df_wells.dropna(subset=['X', 'Y'])
    
    return df_wells

def read_hr_slice(file_path):
    """
    Безопасное считывание файлов HampsonRussell.
    """
    # Имя колонки по умолчанию (имя файла без расширения)
    val_column_name = os.path.splitext(os.path.basename(file_path))[0]
    
    body_started = False
    data_lines = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if '#{BODY}' in line:
                body_started = True
                continue
                
            if body_started:
                cleaned_line = line.strip()
                if not cleaned_line:
                    continue
                
                # Если строка содержит имя атрибута, пытаемся его вытащить
                if 'Value:' in cleaned_line:
                    parts = cleaned_line.split('Value:')
                    if len(parts) > 1:
                        # Убираем знак '>' в конце названия
                        val_column_name = parts[1].replace('>', '').strip()
                    continue
                
                data_lines.append(cleaned_line)
                    
    data_str = '\n'.join(data_lines)
    
    # Читаем как текст, чтобы Pandas не ругался на строки вида "<X> <Y>"
    df_slice = pd.read_csv(io.StringIO(data_str), sep=r'\s+', names=['X', 'Y', 'Value'], dtype=str)
    
    # Конвертируем в числа. Все не-числа станут NaN
    df_slice['X'] = pd.to_numeric(df_slice['X'], errors='coerce')
    df_slice['Y'] = pd.to_numeric(df_slice['Y'], errors='coerce')
    df_slice['Value'] = pd.to_numeric(df_slice['Value'], errors='coerce')
    
    # Очищаем от строк-заголовков (где X или Y превратились в NaN)
    df_slice = df_slice.dropna(subset=['X', 'Y'])
    
    # Исключаем Null Value (-1.0000e+36)
    df_slice = df_slice[df_slice['Value'] > -1e30] 
    
    return df_slice, val_column_name




def extract_values_by_radius(df_wells, hr_files_dir, radius_m):
    """
    Пробегает по всем 60 файлам, строит KDTree и снимает значения для каждой скважины
    """
    # Создаем копию таблицы скважин, куда будем добавлять новые колонки
    df_result = df_wells.copy()
    
    # Координаты скважин для поиска
    well_coords = df_result[['X', 'Y']].values
    
    # Ищем все файлы в папке (можно настроить маску, например '*.txt' или '*.dat')
    file_list = glob.glob(os.path.join(hr_files_dir, '*'))
    
    print(f"Найдено файлов для обработки: {len(file_list)}")
    
    for file_path in file_list:
        try:
            # 1. Читаем файл HR
            df_slice, col_name = read_hr_slice(file_path)
            
            if df_slice.empty:
                print(f"Файл {os.path.basename(file_path)} пустой или содержит только Null Values.")
                df_result[col_name] = np.nan
                continue
                
            # 2. Строим пространственное дерево по координатам из файла HR
            slice_coords = df_slice[['X', 'Y']].values
            tree = KDTree(slice_coords)
            
            # 3. Ищем все точки грида в радиусе radius_m от каждой скважины
            indices_list = tree.query_ball_point(well_coords, r=radius_m)
            
            # 4. Собираем значения
            extracted_values = []
            for i, indices in enumerate(indices_list):
                if indices:
                    # Если точки найдены в радиусе — усредняем их значения (Mean)
                    # При желании np.mean можно заменить на np.median или брать строго ближайшую
                    val = df_slice.iloc[indices]['Value'].mean()
                else:
                    # Если в радиусе пусто, ищем одну самую ближайшую точку без ограничения по радиусу,
                    # либо ставим np.nan (сейчас сделано: пишем NaN)
                    val = np.nan
                    
                extracted_values.append(val)
            
            # Добавляем колонку в итоговый датафрейм
            df_result[col_name] = extracted_values
            print(f"Успешно обработан: {col_name}")
            
        except Exception as e:
            print(f"Ошибка при обработке файла {file_path}: {e}")
            
    return df_result

In [16]:
WELLS_FILE = "./DATA_WELLS/Heff_Ju1.txt"
HR_DIR = './DATA_SLICES/'
SEARCH_RADIUS = 100.0 # Радиус сбора данных в метрах
    
    # 1. Считываем скважины
df_wells = read_well_data(WELLS_FILE)
    
    # 2. Запуск расчета
    # Замените demo_wells на df_wells, когда пропишете реальные пути
final_table = extract_values_by_radius(df_wells, HR_DIR, radius_m=SEARCH_RADIUS)
    
    # 3. Смотрим результат и сохраняем в Excel/CSV
print("\nИтоговая таблица:")
print(final_table.head(10))
    

Найдено файлов для обработки: 60
Успешно обработан: dT_Ju1
Успешно обработан: Envelope_Modulated_Phase_a8b4_raw_amplitude_sum_abs_diff
Успешно обработан: Envelope_Second_Derivative_a8b4_raw_amplitude_max
Успешно обработан: Envelope_Second_Derivative_a8b4_raw_amplitude_min
Успешно обработан: Envelope_Second_Derivative_a8b4_raw_amplitude_rms_average
Успешно обработан: Envelope_Time_Derivative_a8b4_raw_amplitude_mean
Успешно обработан: Envelope_Time_Derivative_a8b4_raw_amplitude_sum_abs_diff
Успешно обработан: Inversion_a8b4_raw_amplitude_mean
Успешно обработан: Ju1_a14b4_2nd_derivative_inst_amp_max
Успешно обработан: Ju1_a14b4_2nd_derivative_inst_amp_mean_negative
Успешно обработан: Ju1_a14b4_2nd_derivative_inst_amp_rms_average
Успешно обработан: Ju1_a14b4_2nd_derivative_inst_amp_sum_abs_diff
Успешно обработан: Ju1_a14b4_2nd_derivative_inst_amp_sum_negative
Успешно обработан: Ju1_a14b4_2nd_derivative_mean
Успешно обработан: Ju1_a14b4_2nd_derivative_median
Успешно обработан: Ju1_a14b4_2nd

In [17]:
final_table.describe()

,X,Y,Heff,dT_Ju1,Envelope_Modulated_Phase_a8b4_raw_amplitude_sum_abs_diff,Envelope_Second_Derivative_a8b4_raw_amplitude_max,Envelope_Second_Derivative_a8b4_raw_amplitude_min,Envelope_Second_Derivative_a8b4_raw_amplitude_rms_average,Envelope_Time_Derivative_a8b4_raw_amplitude_mean,Envelope_Time_Derivative_a8b4_raw_amplitude_sum_abs_diff,...,Ju1_a14b4_phase_mean,Ju1_a14b4_phase_rms_average,Ju1_a14b4_quadrature_trace_max,Ju1_a14b4_raw_amplitude_mean,Ju1_a14b4_raw_amplitude_median,Normalized_Amplitude_a8b4_raw_amplitude_mean,Relative_Acoustic_Impedance_a8b4_raw_amplitude_mean,SD_Envelope_Sub-band_16_a14b4_raw_amplitude_rms_average,SD_Envelope_Sub-band_21_a14b4_raw_amplitude_mean,Trace_Envelope_a14b4_raw_amplitude_sum_abs_diff
count,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,...,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000
mean,325363.307692,783817.564103,5.674615,0.032069,23.257618,736.602203,-4033.083726,2005.389044,-1455.926968,7495.497674,...,24.559403,125.418480,3129.134924,-1157.517193,-1103.782107,-0.378887,20855.910976,3440.958381,3182.303232,24218.203146
std,2462.286754,4199.638924,2.568837,0.005589,8.198794,390.788785,1962.440192,860.211559,1446.394943,3433.093489,...,48.932433,12.878477,1253.133116,515.204373,596.701903,0.429249,24529.184427,1171.443088,1022.936533,7642.949075
min,320131.000000,771924.000000,0.000000,0.023354,9.540231,114.376426,-9274.690091,384.957148,-4117.197688,1633.793017,...,-102.470833,98.841307,228.179998,-2158.075192,-2485.861054,-0.933715,-37021.292885,470.677109,1124.007415,4879.293258
25%,323984.000000,782428.000000,4.760000,0.028202,16.232148,448.112216,-5332.783743,1489.187627,-2408.332888,4700.012657,...,-4.059116,115.634239,2224.056062,-1407.709498,-1472.719610,-0.745214,-271.592347,2621.338312,2374.996935,20436.139105
50%,325592.000000,785537.000000,5.840000,0.030138,23.763722,586.973840,-3943.662256,1807.215336,-1737.531272,7492.642073,...,18.541382,123.857990,3468.382988,-1224.006900,-1219.327151,-0.489158,18436.032640,3684.720619,3369.322239,24419.702708
75%,326593.000000,786544.500000,7.620000,0.034848,29.626809,1018.456548,-2677.390784,2559.029913,-576.578109,9566.608798,...,52.968421,135.737931,4034.999482,-847.413555,-819.993133,-0.105370,37359.485232,4359.305287,3968.308033,29183.242463
max,332670.000000,787741.000000,9.650000,0.045479,41.168669,1720.298135,-532.204468,4329.859961,3442.444681,16245.460783,...,112.763360,149.098479,5283.877623,491.252456,572.256186,0.568419,67089.167080,5430.934467,4755.154844,39723.144870
